# Task: GNN training on synthetic SPV simulations: adjacency, cell state and property matrices

We will be developing a graph neural network (GNN)-based model capable of inferring mechanistic rules and uncovering the principles driving DPAC aggregation. To facilitate this, the GNN will initially be trained using synthetic Self-Propelled Voronoi (SPV) simulations, serving as placeholder data while the deep learning infrastructure is optimized. The GNN will be validated by its ability to, first, recover the physical mechanisms embedded in the SPV model, then subsequently applied to DPAC data to explore the impacts of initial thickness and cell density.

### GNN training

In [ ]:

import networkx as nx
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

from pysr import PySRRegressor
import numpy as np
import torch.nn as nn
from torch.optim import Adam
from torch_geometric.utils import from_networkx, add_self_loops
from tqdm import tqdm
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.nn import MessagePassing
from sklearn.model_selection import KFold


In [2]:
import os
import networkx as nx
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F  # Added for loss functions
from torch.optim import Adam, AdamW, lr_scheduler  # Added AdamW
from torch_geometric.utils import from_networkx
from torch_geometric.data import DataLoader  # Added for data loading
from torch_geometric.nn import MessagePassing
from sklearn.model_selection import KFold

In [ ]:
def parse_parameters(param_path):
    """
    Parse parameters from the given file.
    """
    params = {}
    print(f"Parsing parameters from {param_path}")
    with open(param_path, "r") as f:
        exec(f.read(), {}, params)  # Execute the file content in a controlled namespace
    if not isinstance(params["W"], np.ndarray):
        params["W"] = np.array(params["W"])  # Convert W to NumPy array
    print(f"Successfully parsed parameters: {params}")
    return params

def load_matrices(directory, timepoint):
    """Load matrices with caching for faster subsequent access"""
    cache_key = (directory, timepoint)
    if not hasattr(load_matrices, 'cache'):
        load_matrices.cache = {}
    if cache_key not in load_matrices.cache:
        print(f"Loading matrices for timepoint {timepoint} from {directory}")
        load_matrices.cache[cache_key] = (
            np.load(os.path.join(directory, f"{timepoint}_graph_mat.npy")),
            np.load(os.path.join(directory, f"{timepoint}_properties_mat.npy")),
            np.load(os.path.join(directory, f"{timepoint}_state_mat.npy"))
        )
    return load_matrices.cache[cache_key]

def initialize_gca(graph_mat, properties_mat, state_mat, params):
    """Create graph with proper state handling and parameter validation"""
    print("Initializing GCA graph...")
    g = nx.from_numpy_array(graph_mat, create_using=nx.Graph)
    
    # Validate physical constraints
    if np.any(properties_mat[:, 0] < 0) or np.any(properties_mat[:, 1] < 0):
        raise ValueError("Area and perimeter must be non-negative")

    # Add node properties with proper state handling
    print("Adding node properties...")
    for i, (area, perimeter) in enumerate(properties_mat):
        if area < 0 or perimeter < 0:
            raise ValueError(f"Invalid properties at node {i}: area={area}, perimeter={perimeter}")
        
        cell_type = np.argmax(state_mat[i])
        g.nodes[i]["state"] = state_mat[i]  # Store full state vector
        g.nodes[i].update({
            "area": max(area, 0),
            "perimeter": max(perimeter, 0),
            "motility": params["v0"][cell_type],  # Index into v0 array
            "persistence": params["Dr"],  # Scalar value
            "kappa_A": params["kappa_A"],  # Scalar value
            "kappa_P": params["kappa_P"],  # Scalar value
            "A0": params["A0"][cell_type],  # Index into A0 array
            "P0": params["P0"][cell_type],  # Index into P0 array
        })

    # Add edge properties with validation
    print("Adding edge properties...")
    for u, v in g.edges():
        type_u = np.argmax(g.nodes[u]["state"])
        type_v = np.argmax(g.nodes[v]["state"])
        adhesion = params["W"][type_u][type_v]  # Fetch adhesion from W matrix
        g.edges[u, v].update({
            "adhesion": adhesion,
            "repulsion_radius": params["a"],  # Scalar value
            "repulsion_coefficient": params["k"],  # Scalar value
        })

    g.graph["adj_matrix"] = graph_mat
    print("GCA graph initialization complete.")
    return g

def train_epoch(model, data_loader, optimizer, device, num_cell_types):
    model.train()
    total_loss = 0
    criterion_state = nn.KLDivLoss()
    criterion_prop = nn.SmoothL1Loss()  # More robust than MSE for physical properties
    
    print(f"Starting training epoch with {len(data_loader)} batches...")
    for batch_idx, batch in enumerate(data_loader):
        optimizer.zero_grad()
        
        # Predict next state
        state_pred, area_pred, perim_pred, adj_pred = model(
            batch.x.to(device),
            batch.edge_index.to(device),
            batch.edge_attr.to(device)
        )
        
        # Calculate losses with physical constraints
        loss_state = criterion_state(state_pred, batch.next_state.to(device))
        loss_area = criterion_prop(area_pred, batch.next_area.to(device))
        loss_perim = criterion_prop(perim_pred, batch.next_perim.to(device))
        loss_adj = nn.BCEWithLogitsLoss()(adj_pred, batch.next_adj.to(device))
        
        # Combined loss with regularization
        total_batch_loss = (loss_state + loss_area + loss_perim + loss_adj)
        total_batch_loss += 1e-4 * sum(p.pow(2.0).sum() for p in model.parameters())  # L2 reg
        
        total_batch_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
        optimizer.step()
        
        total_loss += total_batch_loss.item()
        
        if (batch_idx + 1) % 10 == 0:
            print(f"Batch {batch_idx + 1}/{len(data_loader)} - Loss: {total_batch_loss.item():.4f}")
    
    avg_loss = total_loss / len(data_loader)
    print(f"Epoch complete. Average loss: {avg_loss:.4f}")
    return avg_loss

def validate(model, data_loader, device, num_cell_types):
    model.eval()
    total_loss = 0
    print(f"Starting validation with {len(data_loader)} batches...")
    with torch.no_grad():
        for batch_idx, batch in enumerate(data_loader):
            state_pred, area_pred, perim_pred, adj_pred = model(
                batch.x.to(device),
                batch.edge_index.to(device),
                batch.edge_attr.to(device)
            )
            
            # Validation metrics
            loss_state = F.kl_div(state_pred, batch.next_state.to(device), reduction='batchmean')
            loss_area = F.l1_loss(area_pred, batch.next_area.to(device))
            loss_perim = F.l1_loss(perim_pred, batch.next_perim.to(device))
            loss_adj = F.binary_cross_entropy(adj_pred, batch.next_adj.to(device))
            
            total_loss += (loss_state + loss_area + loss_perim + loss_adj).item()
            
            if (batch_idx + 1) % 10 == 0:
                print(f"Validation Batch {batch_idx + 1}/{len(data_loader)} - Loss: {total_loss:.4f}")
    
    avg_loss = total_loss / len(data_loader)
    print(f"Validation complete. Average loss: {avg_loss:.4f}")
    return avg_loss

def k_fold_training(data_dirs, param_files, timepoints, timepoint_interval,
                    node_dim, edge_dim, hidden_dim, num_cell_types,
                    epochs=100, lr=1e-3, k_folds=5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    best_model = None
    best_loss = float('inf')
    
    # Preprocess all data
    print("Preprocessing data...")
    all_data = []
    for dir_idx, (data_dir, param_file) in enumerate(zip(data_dirs, param_files)):
        print(f"Processing directory {data_dir} with parameter file {param_file}...")
        params = parse_parameters(param_file)
        for t in range(0, timepoints - timepoint_interval, timepoint_interval):
            print(f"Loading timepoint {t}...")
            current = load_matrices(data_dir, t)
            next_t = load_matrices(data_dir, t + timepoint_interval)
            
            print(f"Initializing GCA for timepoint {t}...")
            g_current = initialize_gca(*current, params)
            g_next = initialize_gca(*next_t, params)
            
            # Convert to PyG data format with next state info
            data = from_networkx(g_current)
            data.next_state = torch.tensor(np.array([g_next.nodes[i]["state"] for i in g_current.nodes]))
            data.next_area = torch.tensor([g_next.nodes[i]["area"] for i in g_current.nodes])
            data.next_perim = torch.tensor([g_next.nodes[i]["perimeter"] for i in g_current.nodes])
            data.next_adj = torch.tensor(nx.to_numpy_array(g_next)).flatten()
            
            all_data.append(data)
    
    # K-fold training
    kf = KFold(n_splits=k_folds, shuffle=True)
    for fold, (train_idx, val_idx) in enumerate(kf.split(all_data)):
        print(f"\n--- Starting Fold {fold+1}/{k_folds} ---")
        train_data = [all_data[i] for i in train_idx]
        val_data = [all_data[i] for i in val_idx]
        
        print(f"Training data size: {len(train_data)}")
        print(f"Validation data size: {len(val_data)}")
        
        model = GraphPredictor(node_dim, edge_dim, hidden_dim, num_cell_types).to(device)
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5)
        
        # Training loop with early stopping
        best_val_loss = float('inf')
        patience_counter = 0
        for epoch in range(epochs):
            print(f"\nEpoch {epoch+1}/{epochs}")
            train_loss = train_epoch(model, train_data, optimizer, device, num_cell_types)
            val_loss = validate(model, val_data, device, num_cell_types)
            scheduler.step(val_loss)
            
            print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(model.state_dict(), f"best_fold{fold}.pth")
                print(f"New best model saved for fold {fold+1} with validation loss: {best_val_loss:.4f}")
            else:
                patience_counter += 1
                if patience_counter >= 10:
                    print("Early stopping triggered")
                    break
        
        # Update best overall model
        if best_val_loss < best_loss:
            best_loss = best_val_loss
            best_model = model
    
    print(f"\nTraining complete. Best validation loss: {best_loss:.4f}")
    torch.save(best_model.state_dict(), "best_model.pth")
    return best_model

In [ ]:
def main():
    # Configuration
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/{i}_matrix_output"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]

    # Get parameters from first file to determine cell types
    params = parse_parameters(param_files[0])
    num_cell_types = params['W'].shape[0]
    
    # Calculate dimensions based on data structure
    node_dim = num_cell_types + 6  # cell_type one-hot + 6 properties
    edge_dim = 3  # adhesion, repulsion_radius, repulsion_coefficient
    
    # Time configuration for 4-step spacing (0, 4, 8,...)
    max_timepoint = 2000  # Maximum timepoint to consider
    timepoint_step = 4  # Spacing between consecutive timepoints
    
    # Model configuration
    config = {
        'hidden_dim': 64,
        'epochs': 100,
        'learning_rate': 1e-3,
        'k_folds': 3,
        'timepoints': max_timepoint,
        'timepoint_interval': timepoint_step
    }

    # Run training
    best_model = k_fold_training(
        data_dirs=data_dirs,
        param_files=param_files,
        timepoints=config['timepoints'],
        timepoint_interval=config['timepoint_interval'],
        node_dim=node_dim,
        edge_dim=edge_dim,
        hidden_dim=config['hidden_dim'],
        num_cell_types=num_cell_types,
        epochs=config['epochs'],
        lr=config['learning_rate'],
        k_folds=config['k_folds']
    )

    print("Training completed. Best model saved to best_model.pth")

if __name__ == "__main__":
    main()

In [ ]:
if __name__ == "__main__":
    # Directories and parameter files
    data_dirs = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/{i}_matrix_output"
        for i in range(1, 11)
    ]
    param_files = [
        f"/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/DPAC1_parameters/{i}.py"
        for i in range(1, 11)
    ]

    # Run the main training process
    trained_model = main(data_dirs, param_files)

    # Save the final trained model
    torch.save(trained_model.state_dict(), "final_trained_model.pth")
    print("Final model saved to 'final_trained_model.pth'")

In [ ]:
import os

# Check if the files exist
directory = "/Users/sophia01px2019/Downloads/gartner_lab_rotation/GutSPV_tri2/output_Fig5I_matrix/1_matrix_output"
timepoint = 0  # Replace with the correct timepoint
required_files = [
    f"{timepoint}_graph_mat.npy",
    f"{timepoint}_properties_mat.npy",
    f"{timepoint}_state_mat.npy"
]

for file in required_files:
    file_path = os.path.join(directory, file)
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
    else:
        print(f"File exists: {file_path}")